In [1]:
import os
from dotenv import load_dotenv

load_dotenv() 

if os.getenv("LANGSMITH_API_KEY"):
    print("LangSmith API Key loaded")
if os.getenv("GOOGLE_API_KEY"):
    print("Google API Key loaded")

LangSmith API Key loaded
Google API Key loaded


In [2]:
import json
from langchain_core.documents import Document

def load_corpus(file_path):
    with open(file_path, 'r') as f:
        corpus = json.load(f)
    return [
        Document(
            page_content=item["content"],
            metadata={"pmid": item["pmid"], "url": item["url"]}
        ) for item in corpus
    ]
    
documents = load_corpus("./corpus.json")

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview", temperature=0)

# Multi Representation

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

multi_representation_summary_template = "Summarize the following document:\n\n{doc}"

multi_representation_summary_prompt = ChatPromptTemplate.from_template(multi_representation_summary_template)

multi_representation_summary_chain = (
    {"doc": RunnablePassthrough()}
    | multi_representation_summary_prompt
    | llm
    | StrOutputParser()
)

In [16]:
docs = documents[:10]
summaries = []
for doc in docs:
    summaries.append(multi_representation_summary_chain.invoke(doc.page_content))

In [17]:
summaries

['This review explores the **gut-heart axis**, examining how gut microbiota influence cardiovascular health across different species and human populations.\n\n**Key takeaways include:**\n\n*   **The Value of Diverse Models:** While rodent models are essential for identifying biological mechanisms, large-animal models (like swine) are better for replicating human cardiac physiology. Human studies provide the necessary clinical context but are often complicated by high variability.\n*   **Conserved Functional Pathways:** Despite differences in specific microbial species, certain metabolic functions are consistently linked to heart disease across populations. These include the production of short-chain fatty acids, bile acid biotransformation, and the generation of metabolites like TMAO and phenylacetylglutamine.\n*   **The Challenge of Heterogeneity:** Research findings are frequently inconsistent due to population-specific factors, including diet, lifestyle, host genetics (e.g., ABO/FUT

In [30]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
import uuid

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_texts([""], embeddings)
store = InMemoryByteStore()
id_key = "doc_id"

retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
    search_kwargs={"k": 1}
)

doc_ids = [str(uuid.uuid4()) for _ in docs]

summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

retriever.vectorstore.add_documents(summary_docs)
retriever.docstore.mset(list(zip(doc_ids, docs)))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
summary_docs

[Document(metadata={'doc_id': '79736ceb-3f7c-4e68-9049-a8e3c08c59bc'}, page_content='This review explores the **gut-heart axis**, examining how gut microbiota influence cardiovascular health across different species and human populations.\n\n**Key takeaways include:**\n\n*   **The Value of Diverse Models:** While rodent models are essential for identifying biological mechanisms, large-animal models (like swine) are better for replicating human cardiac physiology. Human studies provide the necessary clinical context but are often complicated by high variability.\n*   **Conserved Functional Pathways:** Despite differences in specific microbial species, certain metabolic functions are consistently linked to heart disease across populations. These include the production of short-chain fatty acids, bile acid biotransformation, and the generation of metabolites like TMAO and phenylacetylglutamine.\n*   **The Challenge of Heterogeneity:** Research findings are frequently inconsistent due to p

In [32]:
query = "Antibiotic resistance patterns in community acquired pneumonia"
sub_docs = vectorstore.similarity_search(query, k = 1)
sub_docs

[Document(id='162912b1-5d7e-483e-a633-2d1a5bf6a26e', metadata={'doc_id': '79736ceb-3f7c-4e68-9049-a8e3c08c59bc'}, page_content='This review explores the **gut-heart axis**, examining how gut microbiota influence cardiovascular health across different species and human populations.\n\n**Key takeaways include:**\n\n*   **The Value of Diverse Models:** While rodent models are essential for identifying biological mechanisms, large-animal models (like swine) are better for replicating human cardiac physiology. Human studies provide the necessary clinical context but are often complicated by high variability.\n*   **Conserved Functional Pathways:** Despite differences in specific microbial species, certain metabolic functions are consistently linked to heart disease across populations. These include the production of short-chain fatty acids, bile acid biotransformation, and the generation of metabolites like TMAO and phenylacetylglutamine.\n*   **The Challenge of Heterogeneity:** Research fi

In [33]:
retriever.invoke(query)

[Document(metadata={'pmid': '41520281', 'url': 'https://pubmed.ncbi.nlm.nih.gov/41520281/'}, page_content='Comparative insights into the gut-heart axis: cross-species and cross-population perspectives.Gut microbiota research has rapidly expanded our understanding of host-microbe interactions in cardiovascular diseases, yet translation of these insights remains challenged by species-specific differences and substantial population heterogeneity. In this review, we synthesize current evidence across rodents, swine, non-human primates, and multi-ethnic human cohorts to delineate conserved versus context-dependent features of the gut-heart axis. Rodent models remain indispensable for mechanistic discovery, enabling causal testing through germ-free, antibiotic-treated, and humanized microbiota platforms, whereas large-animal models better replicate human cardiac anatomy, physiology, and microbial ecology. Human studies provide essential clinical relevance, demonstrating that patients with my

In [40]:
retriever.docstore.mget(["79736ceb-3f7c-4e68-9049-a8e3c08c59bc"])

[Document(metadata={'pmid': '41520281', 'url': 'https://pubmed.ncbi.nlm.nih.gov/41520281/'}, page_content='Comparative insights into the gut-heart axis: cross-species and cross-population perspectives.Gut microbiota research has rapidly expanded our understanding of host-microbe interactions in cardiovascular diseases, yet translation of these insights remains challenged by species-specific differences and substantial population heterogeneity. In this review, we synthesize current evidence across rodents, swine, non-human primates, and multi-ethnic human cohorts to delineate conserved versus context-dependent features of the gut-heart axis. Rodent models remain indispensable for mechanistic discovery, enabling causal testing through germ-free, antibiotic-treated, and humanized microbiota platforms, whereas large-animal models better replicate human cardiac anatomy, physiology, and microbial ecology. Human studies provide essential clinical relevance, demonstrating that patients with my